# PhysioGraph extension: circulatory context of SpO2 instability

Locked 2026-09-05 UTC before extraction of the new joint-signal features.
Scope: **MIMIC and eICU only**, original files under My Drive/Data. This develops
the current SpO2-instability project, especially its cross-database myocardial
injury association. No molecular or unrelated datasets are part of this work.

## Biological question

Does an abrupt SpO2 change accompanied by a contemporaneous fall in systemic
arterial pressure distinguish episodes followed by myocardial injury and
death from isolated SpO2 changes? This tests a circulatory-context hypothesis;
it does not assume that a peripheral oximeter measures cardiac output or
microcirculation. The prior analysis already found that recovery rises can
predict events, so absolute changes remain primary and direction is explicit.

## Unchanged anchors

Use the production PhysioGraph adult, explicit-HF phenotype, first ICU per
hospital encounter, early ICU admission or cardiogenic shock, and exclusion
of death or end of observable follow-up by four hours. Follow-up ends at
the earlier of ICU and hospital discharge, matching production code.
Use 0-4-hour SpO2 values in 50-100%,
same-time medians, 15-minute median bins, at least three bins and two
forward transitions no more than 30 minutes apart. The SpO2 exposure remains
an absolute adjacent change of at least four percentage points.

## New feature and comparators

For each qualifying SpO2 transition, obtain MAP medians in its two bins.
Never carry a pressure measurement across an empty bin. A circulatory decline
is a MAP decrease of at least 10 mmHg over that same transition. MAP values
outside 20-200 mmHg are excluded; the threshold is fixed before outcomes.

Classify eligible encounters into four states: neither event, SpO2 change
only, MAP decline only, both. Also record whether the events occur on the
SAME transition; co-occurrence anywhere within four hours is not synchrony.
Require at least two jointly observed transitions. Do not label missing MAP
as stable circulation. Preserve arterial and noninvasive pressure provenance;
eICU currently lacks the noninvasive-pressure source.

Primary question: does synchronous instability carry information beyond
separate SpO2-jump and MAP-decline indicators, their event burdens, absolute
levels, age, sex, heart rate and sampling? Secondary fixed checks: downward
versus upward SpO2 jumps; stable absolute oxygenation (all bins >=90%);
normal measured baseline lactate (<2 mmol/L); invasive-pressure-only MIMIC.
Normal lactate requires an actual measurement; missing is not normal.

## Endpoints and safeguards

Two primary endpoint families: subsequent hospital death and the existing
assay-matched troponin rise endpoint after the four-hour landmark. For the
first feasibility/feature build, extract raw labs through 28 hours and report
coverage before any effect estimates. Freeze the exact window and model
after verifying assay/time support, before new feature-outcome associations.
This document freezes the feature, not a fully specified final model.

An eventual mortality model must compare against separate component signals
and baseline physiology/respiratory treatment, with patient-clustered
uncertainty and eICU hospital heterogeneity. No future lactate, troponin,
ventilation or pressors enter baseline adjustment. Myocardial injury is not
adjudicated infarction. Troponin ordering requires an observation analysis.
Do not claim full SCAI equivalence when source fields are missing.

Minimum support for a primary fitted contrast: 100 endpoint events overall,
at least 20 events and 20 non-events in the synchronous group and comparator,
and adequate model/weight support. Sparse strata remain descriptive. Do not
choose a replacement threshold or endpoint to rescue a failed result. Report
both databases separately, transfer definitions without retuning, and retain
negative results. Previously explored databases are not pristine holdouts.

## Gates for advancing

A robust cross-database association is only the first gate. Useful early
warning also requires improvement over component signals, lactate and
faithfully available shock measures; calibration/net benefit; temporal and
treatment controls; and direct mechanistic corroboration. Mortality reduction
needs intervention evidence. Neither association nor a new score alone
completes the user's paradigm-shifting discovery goal.

In [1]:
CONTEXT_PROTOCOL_SHA256='f7d78bdc2aba420320434e9a72a576b72dfd825353491f0a36d1569467b35b23'

In [2]:
import sys, importlib.util, subprocess, json, hashlib, os
from pathlib import Path
from datetime import datetime, timezone
requirements={'numpy':'numpy==2.2.6','pandas':'pandas==2.2.3','duckdb':'duckdb==1.4.3','yaml':'pyyaml'}
missing=[p for m,p in requirements.items() if importlib.util.find_spec(m) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import numpy as np
import pandas as pd
import duckdb
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE=Path('/content/drive/MyDrive')
else:DRIVE=Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
assert (PROJECT/'src/physiograph').is_dir(), 'Keep the PhysioGraph project beside Data in Drive'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.etl.audit import AuditLogger
from physiograph.etl.mimic_extractor import _load_mimic_cohort, _mimic_chartevent_item_map, _mimic_lab_item_map
from physiograph.etl.eicu_extractor import _load_eicu_patients, _build_eicu_diagnosis_flags, _build_eicu_cohort
from physiograph.constants import EICU_LAB_NAME_MAP, EICU_VITAL_COLUMN_MAP
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
PRIVATE.mkdir(exist_ok=True)
OUT=PROJECT/'research'/'spo2_circulatory_context';OUT.mkdir(parents=True,exist_ok=True)
ENVIRONMENT='google_colab' if IN_COLAB else 'local'
def context_json(name,value):
    (OUT/name).write_text(json.dumps(value,indent=2,allow_nan=False,default=str)+'\n')
def context_status(stage,state='running'):
    context_json('run_status.json',dict(stage=stage,status=state,environment=ENVIRONMENT,
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,state,flush=True)
context_json('protocol.json',dict(sha256=CONTEXT_PROTOCOL_SHA256,frozen_before_feature_extraction=True))
con=duckdb.connect(str(PRIVATE/'spo2_context.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
RAW={d:DRIVE/'Data'/d/'Full' for d in ['MIMIC','eICU']}
sources={}
def qs(value):return "'"+str(value).replace("'","''")+"'"
def raw_csv(dataset,name):
    p=RAW[dataset]/(name+'.csv');s=p.stat()
    sources[dataset+'/'+name]=dict(path=str(p),bytes=s.st_size,mtime_ns=s.st_mtime_ns)
    return f'read_csv({qs(p)},header=true,all_varchar=true,sample_size=10000,strict_mode=true)'
context_status('production cohort reconstruction')
cohorts={};cohort_summary=[]
for d,source in [('eicu','eICU'),('mimic','MIMIC')]:
    audit=AuditLogger(d)
    if d=='eicu' and globals().get('RESUME_EICU',False):
        # Explicit recovery of the successfully committed step from this run.
        c=con.execute('SELECT * FROM eicu_cohort').df()
        assert c.stay_id.is_unique and c.age.ge(18).all() and c.cohort_hf_flag.eq(1).all()
        assert (c.early_icu_flag.eq(1)|c.shock_icd_flag.eq(1)).all()
        print('Resumed committed eICU cohort from the preceding local attempt:',len(c))
        for name in ['patient','diagnosis','admissionDx']:raw_csv(source,name)
    elif d=='eicu':
        patient=_load_eicu_patients(RAW[source],audit)
        flags=_build_eicu_diagnosis_flags(RAW[source],patient)
        c,_=_build_eicu_cohort(RAW[source],patient,flags,audit,max_stays=None)
        outcome=patient.set_index('patientunitstayid').hospitaldischargestatus.str.lower()
        c['hospital_mortality']=c.stay_id.map(outcome.map({'expired':1.,'alive':0.}))
        for name in ['patient','diagnosis','admissionDx']:raw_csv(source,name)
    else:
        c,_,full=_load_mimic_cohort(RAW[source],audit,max_stays=None)
        mortality=pd.read_csv(RAW[source]/'admissions.csv',usecols=['hadm_id','hospital_expire_flag']).set_index('hadm_id').hospital_expire_flag
        c['hospital_mortality']=c.hadm_id.map(mortality).astype(float)
        for name in ['patients','admissions','icustays','diagnoses_icd']:raw_csv(source,name)
    before=len(c)
    exclude=c.death_offset_minutes.between(0,240)|c.followup_end_offset_minutes.le(240)
    c=c.loc[~exclude].copy()
    assert c.stay_id.is_unique
    c['encounter_id']=c['hadm_id'] if d=='mimic' else c['hospital_encounter_id']
    con.register('cohort_frame',c)
    con.execute(f'CREATE OR REPLACE TABLE {d}_cohort AS SELECT * FROM cohort_frame')
    cohorts[d]=c
    cohort_summary.append(dict(dataset=d,before_exclusion=before,eligible=len(c),
        people=c.person_id.nunique(),known_hospital_outcomes=c.hospital_mortality.notna().sum()))
context_json('cohort_support.json',cohort_summary)
print(pd.DataFrame(cohort_summary).to_string(index=False))

production cohort reconstruction running
dataset  before_exclusion  eligible  people  known_hospital_outcomes
   eicu             12304     12028   10647                    11910
  mimic             17892     17758   13464                    17758


In [3]:
context_status('eICU raw vital scan')
source=raw_csv('eICU','vitalPeriodic')
concepts={'sao2':'spo2','systemicmean':'map','heartrate':'hr','systemicsystolic':'sbp','respiration':'resp_rate'}
con.execute(f"""CREATE OR REPLACE TABLE eicu_vitals_wide AS
SELECT try_cast(v.patientunitstayid AS BIGINT) stay_id,
try_cast(v.observationoffset AS DOUBLE) event_minute,
{','.join('try_cast(v.'+col+' AS DOUBLE) AS '+concept for col,concept in concepts.items())}
FROM {source} v JOIN eicu_cohort c ON try_cast(v.patientunitstayid AS BIGINT)=c.stay_id
WHERE try_cast(v.observationoffset AS DOUBLE)>=0 AND try_cast(v.observationoffset AS DOUBLE)<240""")
con.execute("CREATE OR REPLACE TABLE eicu_vitals AS "+' UNION ALL '.join(
    f"SELECT stay_id,event_minute,{qs(concept)} concept,{concept} AS value,'periodic_invasive_context' source_type FROM eicu_vitals_wide WHERE {concept} IS NOT NULL"
    for concept in concepts.values()))
context_status('MIMIC raw vital scan')
itemmap=_mimic_chartevent_item_map(RAW['MIMIC'])
chosen={i:m for i,m in itemmap.items() if m in ['spo2','map','sbp','hr','resp_rate','fio2']}
mapping=pd.DataFrame([dict(itemid=i,concept=m) for i,m in chosen.items()]);con.register('vital_map',mapping)
source=raw_csv('MIMIC','chartevents');raw_csv('MIMIC','d_items')
con.execute(f"""CREATE OR REPLACE TABLE mimic_vitals AS
SELECT c.stay_id,date_diff('second',c.admit_time,try_cast(v.charttime AS TIMESTAMP))/60.0 event_minute,
m.concept,try_cast(v.valuenum AS DOUBLE) AS value,
CASE WHEN try_cast(v.itemid AS BIGINT) IN (220181,220179) THEN 'noninvasive'
WHEN m.concept IN ('map','sbp') THEN 'invasive' ELSE 'charted' END source_type
FROM {source} v JOIN mimic_cohort c ON try_cast(v.stay_id AS BIGINT)=c.stay_id
JOIN vital_map m ON try_cast(v.itemid AS BIGINT)=m.itemid
WHERE try_cast(v.charttime AS TIMESTAMP)>=c.admit_time AND try_cast(v.charttime AS TIMESTAMP)<c.admit_time+INTERVAL '4 hours'
AND coalesce(try_cast(v.warning AS DOUBLE),0)<>1
AND try_cast(v.valuenum AS DOUBLE) IS NOT NULL""")
con.execute('CREATE OR REPLACE TABLE mimic_context_quality_marker AS SELECT 1 AS version')
context_status('joint SpO2 and MAP feature construction')
coverage=[]
for d in ['eicu','mimic']:
    con.execute(f"""CREATE OR REPLACE TABLE {d}_bins AS
    WITH dedup AS (
      SELECT stay_id,event_minute,concept,median(value) AS value FROM {d}_vitals
      WHERE (concept='spo2' AND value BETWEEN 50 AND 100)
         OR (concept='map' AND value BETWEEN 20 AND 200)
         OR (concept='hr' AND value BETWEEN 20 AND 250)
         OR (concept='sbp' AND value BETWEEN 30 AND 300)
         OR (concept='resp_rate' AND value BETWEEN 2 AND 80)
      GROUP BY stay_id,event_minute,concept
    ) SELECT stay_id,floor(event_minute/15)::INTEGER bin,concept,median(value) AS value,
      median(event_minute) AS representative_minute,
      count(*) n_readings FROM dedup GROUP BY stay_id,bin,concept""")
    bins=con.execute(f'SELECT * FROM {d}_bins ORDER BY stay_id,bin').df()
    wide=bins.pivot(index=['stay_id','bin'],columns='concept',values='value').reset_index()
    clock=bins[bins.concept.eq('spo2')][['stay_id','bin','representative_minute']]
    wide=wide.merge(clock,on=['stay_id','bin'],how='left',validate='one_to_one')
    feature_rows=[]
    for stay,s in wide.groupby('stay_id',sort=False):
        s=s[s.spo2.notna()].sort_values('bin')
        gap=s.representative_minute.diff();ds=s.spo2.diff();dm=s['map'].diff()
        transition=gap.gt(0)&gap.le(30)
        joint=transition&dm.notna()
        jump=ds.abs().ge(4)&joint;fall=dm.le(-10)&joint
        joint_n=int(joint.sum());spo2_n=len(s)
        row=dict(stay_id=int(stay),spo2_bins=spo2_n,spo2_transitions=int(transition.sum()),
            joint_transitions=joint_n,joint_eligible=spo2_n>=3 and joint_n>=2,
            spo2_jump_any=int((ds.abs().ge(4)&transition).any()),
            joint_spo2_jump_any=int(jump.any()),map_fall_any=int(fall.any()),
            synchronous_any=int((jump&fall).any()),synchronous_drop=int((jump&fall&ds.lt(0)).any()),
            synchronous_rise=int((jump&fall&ds.gt(0)).any()),
            joint_spo2_jumps=int(jump.sum()),map_falls=int(fall.sum()),
            spo2_mean=float(s.spo2.mean()),spo2_min=float(s.spo2.min()),
            map_mean=float(s['map'].mean()),map_min=float(s['map'].min()),
            hr_mean=float(s.hr.mean()),map_observed_bins=int(s['map'].notna().sum()))
        feature_rows.append(row)
    features=pd.DataFrame(feature_rows)
    con.register('feature_frame',features)
    con.execute(f'CREATE OR REPLACE TABLE {d}_features AS SELECT * FROM feature_frame')
    eligible=features[features.joint_eligible]
    coverage.append(dict(dataset=d,cohort=len(cohorts[d]),spo2_observed=len(features),
        spo2_dynamics=int(((features.spo2_bins>=3)&(features.spo2_transitions>=2)).sum()),
        jointly_observed=len(eligible),synchronous=int(eligible.synchronous_any.sum()),
        simultaneous_drop=int(eligible.synchronous_drop.sum()),
        simultaneous_rise=int(eligible.synchronous_rise.sum())))
context_json('joint_feature_support.json',coverage)
print(pd.DataFrame(coverage).to_string(index=False))
# No feature/outcome association is computed in this feasibility stage.
context_json('source_metadata.json',sources)
context_status('cohort and joint-feature feasibility','complete')
con.close()

eICU raw vital scan running
MIMIC raw vital scan running
joint SpO2 and MAP feature construction running
dataset  cohort  spo2_observed  spo2_dynamics  jointly_observed  synchronous  simultaneous_drop  simultaneous_rise
   eicu   12028          11730          11452              1022           76                 36                 43
  mimic   17758          17172           4711              2846          223                129                106
cohort and joint-feature feasibility complete


In [4]:
# This cell can resume from the completed raw vital/cohort extraction.
# It does not regenerate or alter the locked SpO2/MAP exposure.
import sys, json, hashlib, importlib.util
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import duckdb
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
if IN_COLAB:DRIVE=Path('/content/drive/MyDrive')
else:DRIVE=Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.etl.mimic_extractor import _mimic_lab_item_map
from physiograph.constants import EICU_LAB_NAME_MAP
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
OUT=PROJECT/'research'/'spo2_circulatory_context'
RAW={d:DRIVE/'Data'/d/'Full' for d in ['MIMIC','eICU']}
con=duckdb.connect(str(PRIVATE/'spo2_context.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
sources=json.loads((OUT/'source_metadata.json').read_text())
def context_json(name,value):
    (OUT/name).write_text(json.dumps(value,indent=2,allow_nan=False,default=str)+'\n')
def context_status(stage,state='running'):
    context_json('run_status.json',dict(stage=stage,status=state,
        environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,state,flush=True)
def qs(value):return "'"+str(value).replace("'","''")+"'"
def raw_csv(dataset,name):
    p=RAW[dataset]/(name+'.csv');s=p.stat()
    sources[dataset+'/'+name]=dict(path=str(p),bytes=s.st_size,mtime_ns=s.st_mtime_ns)
    return f'read_csv({qs(p)},header=true,all_varchar=true,sample_size=10000,strict_mode=true)'
context_status('eICU myocardial-injury and baseline lab extraction')
chosen={k:v for k,v in EICU_LAB_NAME_MAP.items() if v in ['troponin_i','troponin_t','lactate','creatinine','ph']}
mapping=pd.DataFrame([dict(raw_name=k,concept=v) for k,v in chosen.items()]);con.register('lab_map',mapping)
source=raw_csv('eICU','lab')
con.execute(f"""CREATE OR REPLACE TABLE eicu_context_labs AS
SELECT c.stay_id,try_cast(l.labresultoffset AS DOUBLE) event_minute,
try_cast(l.labresultrevisedoffset AS DOUBLE) available_minute,
m.concept,l.labname raw_name,lower(trim(coalesce(l.labmeasurenamesystem,l.labmeasurenameinterface,''))) unit,
try_cast(l.labresult AS DOUBLE) AS value
FROM {source} l JOIN eicu_cohort c ON try_cast(l.patientunitstayid AS BIGINT)=c.stay_id
JOIN lab_map m ON lower(trim(l.labname))=m.raw_name
WHERE try_cast(l.labresultoffset AS DOUBLE) BETWEEN 0 AND least(1680,c.followup_end_offset_minutes)
AND try_cast(l.labresult AS DOUBLE) IS NOT NULL""")
context_status('MIMIC myocardial-injury and baseline lab extraction')
chosen={k:v for k,v in _mimic_lab_item_map(RAW['MIMIC']).items() if v in ['troponin_i','troponin_t','lactate','creatinine','ph']}
mapping=pd.DataFrame([dict(itemid=k,concept=v) for k,v in chosen.items()]);con.register('lab_map',mapping)
source=raw_csv('MIMIC','labevents');raw_csv('MIMIC','d_labitems')
con.execute(f"""CREATE OR REPLACE TABLE mimic_context_labs AS
SELECT c.stay_id,date_diff('second',c.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0 event_minute,
date_diff('second',c.admit_time,try_cast(l.storetime AS TIMESTAMP))/60.0 available_minute,
m.concept,l.itemid raw_name,lower(trim(coalesce(l.valueuom,''))) unit,try_cast(l.valuenum AS DOUBLE) AS value
FROM {source} l JOIN mimic_cohort c ON try_cast(l.hadm_id AS BIGINT)=c.hadm_id
JOIN lab_map m ON try_cast(l.itemid AS BIGINT)=m.itemid
WHERE try_cast(l.charttime AS TIMESTAMP)>=c.admit_time
AND date_diff('second',c.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0<=least(1680,c.followup_end_offset_minutes)
AND try_cast(l.valuenum AS DOUBLE) IS NOT NULL""")
support=[];unit_rows=[]
for d in ['eicu','mimic']:
    c=con.execute(f'SELECT * FROM {d}_cohort').df().set_index('stay_id')
    labs=con.execute(f'SELECT * FROM {d}_context_labs').df()
    bounds={'troponin_t':(0,1000000),'troponin_i':(0,1000000),'lactate':(.1,40),'creatinine':(.1,30),'ph':(6.5,8)}
    keep=np.zeros(len(labs),dtype=bool)
    for concept,(lo,hi) in bounds.items():keep|=labs.concept.eq(concept)&labs.value.between(lo,hi)
    labs=labs[keep].copy()
    unit_rows.extend(labs.groupby(['concept','raw_name','unit']).size().rename('records').reset_index().assign(dataset=d).to_dict('records'))
    labs=labs.groupby(['stay_id','event_minute','concept','unit'],as_index=False).agg(value=('value','median'),available_minute=('available_minute','max'))
    base=labs[labs.event_minute.le(240)].sort_values('event_minute')
    for marker in ['lactate','creatinine','ph']:
        last=base[base.concept.eq(marker)].drop_duplicates('stay_id',keep='last').set_index('stay_id')
        c['baseline_'+marker]=last.value
        c['baseline_'+marker+'_available_minute']=last.available_minute
    base_t=base[base.concept.str.startswith('troponin')&base.value.gt(0)].drop_duplicates(['stay_id','concept','unit'],keep='last')
    for end,label in [(960,'12h'),(1680,'24h')]:
        post=labs[labs.event_minute.gt(240)&labs.event_minute.le(end)&labs.concept.str.startswith('troponin')]
        peak=post.groupby(['stay_id','concept','unit'],as_index=False).value.max().rename(columns={'value':'post_peak'})
        pairs=base_t.merge(peak,on=['stay_id','concept','unit'],how='inner')
        pairs['ratio']=pairs.post_peak/pairs.value
        result=pairs.groupby('stay_id').ratio.max()
        c['troponin_ratio_'+label]=result
        observed=c['troponin_ratio_'+label].notna()
        positive=c['troponin_ratio_'+label].ge(1.5)
        full=c.followup_end_offset_minutes.ge(end)
        c['troponin_rise_'+label]=positive.astype(float).where(observed&(positive|full))
        support.append(dict(dataset=d,endpoint='troponin_'+label,cohort=len(c),
            baseline_positive_assay_people=int(base_t.stay_id.nunique()),
            paired=int(observed.sum()),classifiable=int(c['troponin_rise_'+label].notna().sum()),
            events=int(c['troponin_rise_'+label].sum())))
    con.register('endpoint_frame',c.reset_index())
    con.execute(f'CREATE OR REPLACE TABLE {d}_context_endpoints AS SELECT * FROM endpoint_frame')
context_json('lab_assay_unit_audit.json',unit_rows)
context_json('endpoint_support.json',support)
print(pd.DataFrame(support).to_string(index=False))
context_status('raw source fingerprint verification')
for key,info in sources.items():
    p=Path(info['path']);digest=hashlib.sha256()
    with p.open('rb') as f:
        for block in iter(lambda:f.read(8*1024*1024),b''):digest.update(block)
    s=p.stat()
    assert s.st_size==info['bytes'] and s.st_mtime_ns==info['mtime_ns'], 'Source changed: '+key
    info['sha256']=digest.hexdigest()
context_json('input_manifest.json',sources)
context_status('SpO2-context features and endpoints extracted','complete')
con.close()
print('Feasibility outputs only: no new feature-outcome association or biological discovery claimed.')

eICU myocardial-injury and baseline lab extraction running
MIMIC myocardial-injury and baseline lab extraction running
dataset     endpoint  cohort  baseline_positive_assay_people  paired  classifiable  events
   eicu troponin_12h   12028                            3458    2539          2405     605
   eicu troponin_24h   12028                            3458    2648          2215     668
  mimic troponin_12h   17758                            4485    2858          2781     674
  mimic troponin_24h   17758                            4485    3033          2624     804
raw source fingerprint verification running
SpO2-context features and endpoints extracted complete
Feasibility outputs only: no new feature-outcome association or biological discovery claimed.


In [5]:
# Reconcile the historical recovery attempt to the exact production risk set.
# Raw scans can contain a superset; model tables must use the final cohort.
import json, importlib.util, sys
from pathlib import Path
from datetime import datetime, timezone
import duckdb
import pandas as pd
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
OUT=PROJECT/'research'/'spo2_circulatory_context'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.analysis.spo2_protocol import _first_instability
(OUT/'run_status.json').write_text(json.dumps(dict(stage='production cohort and SpO2 boundary reconciliation',
    status='running',environment='google_colab' if IN_COLAB else 'local',
    updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False),indent=2)+'\n')
con=duckdb.connect(str(DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'/'spo2_context.duckdb'))
con.execute('SET threads=2')
con.execute("SET memory_limit='2GB'")
# Recover the initial scan's omitted production warning filter exactly once.
# Fresh full-notebook extraction above already applies and records this rule.
quality_done=con.execute("SELECT count(*) FROM information_schema.tables WHERE table_name='mimic_context_quality_marker'").fetchone()[0]>0
if not quality_done:
    from physiograph.etl.mimic_extractor import _mimic_chartevent_item_map
    root=DRIVE/'Data'/'MIMIC'/'Full';raw_path=root/'chartevents.csv'
    input_info=json.loads((OUT/'input_manifest.json').read_text())['MIMIC/chartevents']
    st=raw_path.stat()
    assert st.st_size==input_info['bytes'] and st.st_mtime_ns==input_info['mtime_ns'], 'Raw input changed before quality correction'
    (OUT/'run_status.json').write_text(json.dumps(dict(stage='MIMIC production warning-filter correction',
        status='running',environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False),indent=2)+'\n')
    itemmap=_mimic_chartevent_item_map(root)
    mapping=pd.DataFrame([dict(itemid=i,concept=m) for i,m in itemmap.items()
                          if m in ['spo2','map','sbp','hr','resp_rate','fio2']])
    con.register('quality_vital_map',mapping)
    before_quality=con.execute('SELECT count(*) FROM mimic_vitals').fetchone()[0]
    con.execute("""CREATE OR REPLACE TABLE mimic_vitals AS
      SELECT c.stay_id,date_diff('second',c.admit_time,try_cast(v.charttime AS TIMESTAMP))/60.0 event_minute,
      m.concept,try_cast(v.valuenum AS DOUBLE) AS value,
      CASE WHEN try_cast(v.itemid AS BIGINT) IN (220181,220179) THEN 'noninvasive'
      WHEN m.concept IN ('map','sbp') THEN 'invasive' ELSE 'charted' END source_type
      FROM read_csv(?,header=true,all_varchar=true,sample_size=10000,strict_mode=true) v
      JOIN mimic_cohort c ON try_cast(v.stay_id AS BIGINT)=c.stay_id
      JOIN quality_vital_map m ON try_cast(v.itemid AS BIGINT)=m.itemid
      WHERE try_cast(v.charttime AS TIMESTAMP)>=c.admit_time
        AND try_cast(v.charttime AS TIMESTAMP)<c.admit_time+INTERVAL '4 hours'
        AND coalesce(try_cast(v.warning AS DOUBLE),0)<>1
        AND try_cast(v.valuenum AS DOUBLE) IS NOT NULL""",[str(raw_path)])
    con.execute("""CREATE OR REPLACE TABLE mimic_bins AS
      WITH dedup AS (
        SELECT stay_id,event_minute,concept,median(value) AS value FROM mimic_vitals
        WHERE (concept='spo2' AND value BETWEEN 50 AND 100)
          OR (concept='map' AND value BETWEEN 20 AND 200)
          OR (concept='hr' AND value BETWEEN 20 AND 250)
          OR (concept='sbp' AND value BETWEEN 30 AND 300)
          OR (concept='resp_rate' AND value BETWEEN 2 AND 80)
        GROUP BY stay_id,event_minute,concept)
      SELECT stay_id,floor(event_minute/15)::INTEGER bin,concept,median(value) AS value,
        median(event_minute) representative_minute,count(*) n_readings
        FROM dedup GROUP BY stay_id,bin,concept""")
    st=raw_path.stat()
    assert st.st_size==input_info['bytes'] and st.st_mtime_ns==input_info['mtime_ns'], 'Raw input changed during quality correction'
    con.execute('CREATE OR REPLACE TABLE mimic_context_quality_marker AS SELECT 1 AS version')
    (OUT/'mimic_quality_correction.json').write_text(json.dumps(dict(
        raw_rows_before=before_quality,raw_rows_after=con.execute('SELECT count(*) FROM mimic_vitals').fetchone()[0],
        correction='canonical cohort, [0,240) window, and warning !=1',
        production_filter='src/physiograph/etl/mimic_extractor.py:_stream_mimic_measurement_events',
        source_sha256=input_info['sha256'],cross_tabulations_previously_seen=True,
        no_effect_estimates_fitted=True),indent=2)+'\n')
reconciliation=[];support=[]
for d in ['eicu','mimic']:
    before=con.execute(f'SELECT count(*) FROM {d}_cohort').fetchone()[0]
    con.execute(f"""CREATE TABLE IF NOT EXISTS {d}_canonical_exclusions AS
    SELECT * FROM {d}_cohort WHERE FALSE""")
    con.execute(f"""INSERT INTO {d}_canonical_exclusions
    SELECT * FROM {d}_cohort WHERE death_offset_minutes BETWEEN 0 AND 240
      OR followup_end_offset_minutes<=240""")
    con.execute(f"DELETE FROM {d}_cohort WHERE stay_id IN (SELECT stay_id FROM {d}_canonical_exclusions)")
    for suffix in ['features','context_endpoints']:
        con.execute(f'DELETE FROM {d}_{suffix} WHERE stay_id NOT IN (SELECT stay_id FROM {d}_cohort)')
    # Production SpO2 dynamics use [0,240), unlike baseline labs which include 240.
    # The initial raw scan was an inclusive superset; remove its terminal bin.
    boundary_records=con.execute(f'SELECT count(*) FROM {d}_bins WHERE bin>=16').fetchone()[0]
    before_dynamics=con.execute(f'SELECT count(*) FROM {d}_features WHERE spo2_bins>=3 AND spo2_transitions>=2').fetchone()[0]
    con.execute(f'DELETE FROM {d}_bins WHERE bin>=16')
    con.execute(f"""CREATE OR REPLACE TABLE {d}_features AS
      WITH wide AS (
        SELECT b.stay_id,bin,max(value) FILTER(WHERE concept='spo2') spo2,
        max(value) FILTER(WHERE concept='map') map_value,
        max(value) FILTER(WHERE concept='hr') hr,
        max(representative_minute) FILTER(WHERE concept='spo2') spo2_time
        FROM {d}_bins b JOIN {d}_cohort c USING(stay_id) GROUP BY b.stay_id,bin), lagged AS (
        SELECT *,spo2-lag(spo2) OVER w ds,map_value-lag(map_value) OVER w dm,
        spo2_time-lag(spo2_time) OVER w gap FROM wide WHERE spo2 IS NOT NULL
        WINDOW w AS (PARTITION BY stay_id ORDER BY bin)), marked AS (
        SELECT *,coalesce(gap>0 AND gap<=30,FALSE) AS valid_transition,
        coalesce(gap>0 AND gap<=30 AND dm IS NOT NULL,FALSE) joint FROM lagged)
      SELECT stay_id,count(*) spo2_bins,count(*) FILTER(WHERE valid_transition) spo2_transitions,
      count(*) FILTER(WHERE joint) joint_transitions,
      count(*)>=3 AND count(*) FILTER(WHERE joint)>=2 joint_eligible,
      max(CASE WHEN valid_transition AND abs(ds)>=4 THEN 1 ELSE 0 END) spo2_jump_any,
      max(CASE WHEN joint AND abs(ds)>=4 THEN 1 ELSE 0 END) joint_spo2_jump_any,
      max(CASE WHEN joint AND dm<=-10 THEN 1 ELSE 0 END) map_fall_any,
      max(CASE WHEN joint AND abs(ds)>=4 AND dm<=-10 THEN 1 ELSE 0 END) synchronous_any,
      max(CASE WHEN joint AND ds<=-4 AND dm<=-10 THEN 1 ELSE 0 END) synchronous_drop,
      max(CASE WHEN joint AND ds>=4 AND dm<=-10 THEN 1 ELSE 0 END) synchronous_rise,
      count(*) FILTER(WHERE joint AND abs(ds)>=4) joint_spo2_jumps,
      count(*) FILTER(WHERE joint AND dm<=-10) map_falls,
      avg(spo2) spo2_mean,min(spo2) spo2_min,avg(map_value) map_mean,min(map_value) map_min,
      avg(hr) hr_mean,count(map_value) map_observed_bins FROM marked GROUP BY stay_id""")
    c=con.execute(f'SELECT * FROM {d}_cohort').df()
    f=con.execute(f'SELECT * FROM {d}_features').df()
    assert c.stay_id.is_unique and c.age.ge(18).all() and c.cohort_hf_flag.eq(1).all()
    assert (c.early_icu_flag.eq(1)|c.shock_icd_flag.eq(1)).all()
    assert not (c.death_offset_minutes.between(0,240)|c.followup_end_offset_minutes.le(240)).any()
    assert set(f.stay_id).issubset(set(c.stay_id))
    raw_spo2=con.execute(f"""SELECT v.stay_id,'{d}' dataset,concept,
        event_minute offset_minutes,value value_numeric FROM {d}_vitals v
        JOIN {d}_cohort c USING(stay_id) WHERE concept='spo2'""").df()
    production_exposed=set(_first_instability(raw_spo2).stay_id)
    rebuilt_exposed=set(f.loc[f.spo2_bins.ge(3)&f.spo2_transitions.ge(2)&f.spo2_jump_any.eq(1),'stay_id'])
    assert production_exposed==rebuilt_exposed, 'Production SpO2 exposure mismatch: '+d
    eligible=f[f.joint_eligible]
    reconciliation.append(dict(dataset=d,before=before,canonical=len(c),removed_this_execution=before-len(c),
        terminal_bins_removed=boundary_records,dynamics_before_boundary_fix=before_dynamics,
        dynamics_after_boundary_fix=int(((f.spo2_bins>=3)&(f.spo2_transitions>=2)).sum()),
        production_exposure_patient_sets_match=True,production_exposed=len(production_exposed),
        cumulative_excluded=con.execute(f'SELECT count(*) FROM {d}_canonical_exclusions').fetchone()[0],
        historical_reference={'eicu':12028,'mimic':17758}[d],
        matches_historical_count=len(c)=={'eicu':12028,'mimic':17758}[d]))
    support.append(dict(dataset=d,cohort=len(c),spo2_observed=len(f),
        spo2_dynamics=int(((f.spo2_bins>=3)&(f.spo2_transitions>=2)).sum()),
        jointly_observed=len(eligible),synchronous=int(eligible.synchronous_any.sum()),
        simultaneous_drop=int(eligible.synchronous_drop.sum()),simultaneous_rise=int(eligible.synchronous_rise.sum())))
(OUT/'canonical_cohort_reconciliation.json').write_text(json.dumps(reconciliation,indent=2)+'\n')
(OUT/'joint_feature_support_canonical.json').write_text(json.dumps(support,indent=2)+'\n')
print(pd.DataFrame(reconciliation).to_string(index=False))
print(pd.DataFrame(support).to_string(index=False))
con.close()

dataset  before  canonical  removed_this_execution  terminal_bins_removed  dynamics_before_boundary_fix  dynamics_after_boundary_fix  production_exposure_patient_sets_match  production_exposed  cumulative_excluded  historical_reference  matches_historical_count
   eicu   12028      12028                       0                      0                         11452                        11452                                    True                4309                    0                 12028                      True
  mimic   17758      17758                       0                      0                          4711                         4711                                    True                1398                   18                 17758                      True
dataset  cohort  spo2_observed  spo2_dynamics  jointly_observed  synchronous  simultaneous_drop  simultaneous_rise
   eicu   12028          11572          11452              1022           76                 36    

In [6]:
# Pressure-source sensitivity and timing support, without outcome associations.
import json, importlib.util
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
OUT=PROJECT/'research'/'spo2_circulatory_context'
con=duckdb.connect(str(DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'/'spo2_context.duckdb'))
con.execute('SET threads=2')
summaries=[];provenance=[]
for d in ['eicu','mimic']:
    provenance.extend(con.execute(f"""SELECT '{d}' dataset,source_type,count(*) records,
        count(DISTINCT v.stay_id) stays FROM {d}_vitals v JOIN {d}_cohort c USING(stay_id)
        WHERE concept='map' AND value BETWEEN 20 AND 200 GROUP BY source_type""").df().to_dict('records'))
    for variant in (['pooled','arterial'] if d=='mimic' else ['pooled']):
        if variant=='pooled':
            b=con.execute(f'SELECT b.* FROM {d}_bins b JOIN {d}_cohort c USING(stay_id)').df()
        else:
            b=con.execute(f"""WITH dedup AS (
              SELECT v.stay_id,event_minute,concept,median(value) AS value
              FROM {d}_vitals v JOIN {d}_cohort c USING(stay_id)
              WHERE event_minute<240 AND ((concept='spo2' AND value BETWEEN 50 AND 100)
                OR (concept='map' AND source_type='invasive' AND value BETWEEN 20 AND 200)
                OR (concept='hr' AND value BETWEEN 20 AND 250))
              GROUP BY v.stay_id,event_minute,concept)
              SELECT stay_id,floor(event_minute/15)::INTEGER bin,concept,
              median(value) AS value,median(event_minute) representative_minute,
              count(*) n_readings FROM dedup GROUP BY stay_id,bin,concept""").df()
        wide=b.pivot(index=['stay_id','bin'],columns='concept',values='value').reset_index()
        for concept in ['spo2','map']:
            clock=b[b.concept.eq(concept)][['stay_id','bin','representative_minute','n_readings']]
            clock=clock.rename(columns={'representative_minute':concept+'_time','n_readings':concept+'_readings'})
            wide=wide.merge(clock,on=['stay_id','bin'],how='left',validate='one_to_one')
        s=wide[wide.spo2.notna()].sort_values(['stay_id','bin']).copy()
        g=s.groupby('stay_id',sort=False)
        gap=g.spo2_time.diff();ds=g.spo2.diff();dm=g['map'].diff()
        s['valid']=gap.gt(0)&gap.le(30);s['joint']=s.valid&dm.notna()
        s['jump']=ds.abs().ge(4)&s.joint;s['fall']=dm.le(-10)&s.joint
        s['sync']=s.jump&s.fall;s['sync_drop']=s.sync&ds.lt(0);s['sync_rise']=s.sync&ds.gt(0)
        s['low_map']=s['map'].lt(65)
        f=s.groupby('stay_id',sort=False).agg(
            spo2_bins=('spo2','count'),spo2_transitions=('valid','sum'),joint_transitions=('joint','sum'),
            joint_spo2_jump_any=('jump','max'),map_fall_any=('fall','max'),
            synchronous_any=('sync','max'),synchronous_drop=('sync_drop','max'),synchronous_rise=('sync_rise','max'),
            joint_spo2_jumps=('jump','sum'),map_falls=('fall','sum'),
            spo2_mean=('spo2','mean'),spo2_min=('spo2','min'),map_mean=('map','mean'),map_min=('map','min'),
            hr_mean=('hr','mean'),spo2_readings=('spo2_readings','sum'),map_readings=('map_readings','sum'),
            low_map_bins=('low_map','sum'),map_observed_bins=('map','count')).reset_index()
        f['joint_eligible']=f.spo2_bins.ge(3)&f.joint_transitions.ge(2)
        for col in ['joint_spo2_jump_any','map_fall_any','synchronous_any','synchronous_drop','synchronous_rise']:
            f[col]=f[col].astype(int)
        t=pd.DataFrame(dict(stay_id=s.stay_id,bin=s.bin,event_minute=s.spo2_time,
            spo2_change=ds,map_change=dm,joint=s.joint,spo2_jump=s.jump,map_fall=s.fall,
            map_spo2_time_difference=s.map_time-s.spo2_time))
        t=t[t.joint].copy()
        # Independent SQL reconstruction on actual data checks the transition logic.
        con.register('check_bins',b)
        check=con.execute("""WITH wide AS (
          SELECT stay_id,bin,max(value) FILTER(WHERE concept='spo2') spo2,
          max(value) FILTER(WHERE concept='map') map_value,
          max(representative_minute) FILTER(WHERE concept='spo2') spo2_time
          FROM check_bins GROUP BY stay_id,bin), lagged AS (
          SELECT *,spo2-lag(spo2) OVER w ds,map_value-lag(map_value) OVER w dm,
          spo2_time-lag(spo2_time) OVER w gap FROM wide WHERE spo2 IS NOT NULL
          WINDOW w AS (PARTITION BY stay_id ORDER BY bin)), marked AS (
          SELECT *,coalesce(gap>0 AND gap<=30 AND dm IS NOT NULL,FALSE) joint
          FROM lagged)
          SELECT stay_id,count(*) spo2_bins,count(*) FILTER(WHERE joint) joint_transitions,
          count(*) FILTER(WHERE joint AND abs(ds)>=4) joint_spo2_jumps,
          count(*) FILTER(WHERE joint AND dm<=-10) map_falls,
          max(CASE WHEN joint AND abs(ds)>=4 AND dm<=-10 THEN 1 ELSE 0 END) synchronous_any
          FROM marked GROUP BY stay_id""").df().set_index('stay_id')
        actual=f.set_index('stay_id').reindex(check.index)
        for col in check.columns:
            assert actual[col].eq(check[col]).all(), 'Independent SQL feature mismatch: '+d+'/'+variant+'/'+col
        con.register('source_features',f);con.register('source_transitions',t)
        con.execute(f'CREATE OR REPLACE TABLE {d}_{variant}_features AS SELECT * FROM source_features')
        con.execute(f'CREATE OR REPLACE TABLE {d}_{variant}_transitions AS SELECT * FROM source_transitions')
        if variant=='pooled':
            old=con.execute(f'SELECT * FROM {d}_features WHERE spo2_bins>0').df().set_index('stay_id')
            new=f.set_index('stay_id').reindex(old.index)
            for col in ['joint_eligible','spo2_bins','spo2_transitions','joint_transitions',
                        'joint_spo2_jump_any','map_fall_any','synchronous_any','synchronous_drop','synchronous_rise']:
                assert new[col].eq(old[col]).all(), 'Feature definition drift: '+d+'/'+col
        a=f[f.joint_eligible];both=a.joint_spo2_jump_any.eq(1)&a.map_fall_any.eq(1)
        summaries.append(dict(dataset=d,variant=variant,jointly_observed=len(a),
            both_component_events=int(both.sum()),synchronous=int(a.synchronous_any.sum()),
            asynchronous_both=int((both&a.synchronous_any.eq(0)).sum()),
            joint_transitions=len(t),
            median_absolute_within_bin_time_difference=float(t.map_spo2_time_difference.abs().median())))
(OUT/'pressure_provenance.json').write_text(json.dumps(provenance,indent=2)+'\n')
(OUT/'pressure_source_support.json').write_text(json.dumps(summaries,indent=2,allow_nan=False)+'\n')
print(pd.DataFrame(summaries).to_string(index=False))
con.close()

dataset  variant  jointly_observed  both_component_events  synchronous  asynchronous_both  joint_transitions  median_absolute_within_bin_time_difference
   eicu   pooled              1022                    185           76                109              11766                                         0.0
  mimic   pooled              2846                    477          223                254              17645                                         0.0
  mimic arterial              1061                    105           51                 54               5612                                         0.0


# Circulatory-context analysis specification

Written before new exposure–outcome cross-tabulations or model fits on
2026-09-05 UTC. The raw scans are running. Feature and total endpoint
availability may be inspected; no effect estimates have been inspected.
This supplements SPO2_CIRCULATORY_CONTEXT_PLAN.md without changing its
thresholds, cohort, or biological question.

## Estimand and support gate

The primary timing comparison is **synchronous versus asynchronous events
among encounters having both a qualifying SpO2 change and a qualifying MAP
fall during the first four hours**. It is not synchrony versus every other
patient. The latter would mostly compare different component abnormalities
and would require impossible counterfactual synchrony in patients without
either component. Report the four component states descriptively.

Primary outcomes are hospital mortality and the existing troponin-rise
endpoint during 12 hours after the four-hour landmark (ICU minutes 240–960).
The 24-hour endpoint is a sensitivity analysis; a negative or sparse 12-hour
result cannot be rescued by selecting 24 hours. Troponin requires a positive
baseline value and a subsequent value of the same assay and unit, with rise
ratio at least 1.5. Without a detected rise, follow-up must extend through
the horizon. Missing or incompletely observed outcomes are not negatives.

Evaluate the previously frozen event gate **within the primary timing
comparison**, separately in each database: at least 100 events overall and
at least 20 events and 20 non-events in each timing group. Otherwise produce
counts only, explicitly fail the primary fitting gate, and do not lower it.
Also report marginal event support in all jointly observed patients to make
the source of attrition clear. No group with fewer than five patients is
used to support a biological claim.

## Model sequence if support is adequate

First quantify a conditional association with a binomial logit model.
The base model contains age, sex, mean heart rate, mean and minimum SpO2,
mean and minimum MAP, counts of SpO2 jumps and MAP falls, count of jointly
observed transitions, and log(1 + raw SpO2 reading count). The extended
model adds the binary synchrony indicator. Numeric predictors are centered
and scaled within each source; sex is categorical. Missing heart rate uses
the source median and an explicit missing indicator. Other required
physiological values must be observed. No predictor selection by p-value.

Use person-clustered covariance within each source, and an eICU sensitivity
with hospital-clustered covariance when at least 30 hospitals contribute.
Check model convergence, design rank, complete separation, and adequate
events for the number of fitted coefficients (at least 10 events and 10
non-events per coefficient). Failure means no model claim. Report the
conditional odds ratio and 95% confidence interval, explicitly as an odds
ratio. Do not relabel it as a risk ratio or causal effect. Correct the two
primary endpoint tests by Holm within each database. Replication requires
the same direction and multiplicity-adjusted evidence in both sources.

This first model is a **screen**, not the final biological test. Advancement
requires baseline respiratory support, oxygen therapy and vasoactive
treatment controls from original sources, observed baseline lactate and
creatinine, pressure-source sensitivity, ordering/measurement selection
analysis for troponin, and evaluation of predictive improvement. No
unadjusted or screen-only association is promoted as a discovery.

## Fixed sensitivity checks

- Repeat MIMIC features using arterial MAP only. eICU's supplied periodic
  pressure source is arterial; do not infer unmeasured cuff pressure.
- Analyze downward and upward SpO2 changes separately without choosing a
  preferred direction after seeing results.
- Evaluate the predeclared all-SpO2-bins-at-least-90% and measured baseline
  lactate-below-2 mmol/L subgroups only if they pass the same support gate.
  Retrospective specimen-time and actual result-availability definitions
  must be distinguished. eICU revision time is not proven first availability.
- Compare within-bin measurement time differences across pressure sources.
  Same-bin synchrony is coarse charted synchrony, not beat-to-beat coupling.

Low perfusion can alter pulse-oximeter measurement. Neither synchrony nor
troponin elevation establishes myocardial hypoxia, reduced cardiac output,
or a treatable mechanism. Reject significance that depends on a changed
threshold, an unsupported subgroup, a missing-data convention, or a single
database. The paradigm-shifting discovery goal remains unfulfilled unless
substantially stronger evidence is actually obtained.

## Prior-art check

The joint prognostic relevance of low saturation and low blood pressure in
acute HF was already reported in a [2009 clinical cohort](https://pubmed.ncbi.nlm.nih.gov/19372679/).
Combining those two absolute levels is therefore not a new biological finding.
A [2021 registered protocol](https://cdn.clinicaltrials.gov/large-docs/01/NCT05188001/Prot_SAP_000.pdf)
also proposed vital-sign time series for predicting troponin and myocardial
injury after noncardiac surgery; this is adjacent prior art, not an HF result
or proof that its planned analyses succeeded. The narrower timing contrast
in this extension remains a hypothesis. A search finding no exact match
does not establish novelty.

In [7]:
ANALYSIS_LOCK_SHA256='8fc4435812e6456585732f940a4a97e8c83fe86551226475abea33e142582dba'

In [8]:
# Apply the frozen support gate before estimating any association.
import json, importlib.util
from pathlib import Path
from datetime import datetime, timezone
import duckdb
import pandas as pd
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
OUT=PROJECT/'research'/'spo2_circulatory_context'
lock_path=OUT/'analysis_lock.json'
if lock_path.exists():
    assert json.loads(lock_path.read_text())['sha256']==ANALYSIS_LOCK_SHA256, 'Analysis specification changed after locking'
else:
    lock_path.write_text(json.dumps(dict(sha256=ANALYSIS_LOCK_SHA256,
        recorded_before_cross_tabulations=True,utc=datetime.now(timezone.utc).isoformat()),indent=2)+'\n')
con=duckdb.connect(str(DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'/'spo2_context.duckdb'))
gates=[];states=[];endpoint_support=[]
for d in ['eicu','mimic']:
    endpoints=con.execute(f'SELECT * FROM {d}_context_endpoints').df()
    for endpoint in ['hospital_mortality','troponin_rise_12h','troponin_rise_24h']:
        endpoint_support.append(dict(dataset=d,endpoint=endpoint,cohort=len(endpoints),
            observed=int(endpoints[endpoint].notna().sum()),events=int(endpoints[endpoint].sum())))
    for variant in (['pooled','arterial'] if d=='mimic' else ['pooled']):
        f=con.execute(f'SELECT * FROM {d}_{variant}_features WHERE joint_eligible').df()
        a=f.merge(endpoints,on='stay_id',validate='one_to_one')
        a['component_state']=a.joint_spo2_jump_any.astype(int)+2*a.map_fall_any.astype(int)
        for endpoint in ['hospital_mortality','troponin_rise_12h','troponin_rise_24h']:
            observed=a[a[endpoint].notna()]
            for state,g in observed.groupby('component_state'):
                states.append(dict(dataset=d,variant=variant,endpoint=endpoint,
                    component_state={0:'neither',1:'spo2_only',2:'map_only',3:'both'}[state],
                    n=len(g),events=int(g[endpoint].sum())))
            for subgroup in ['all','all_spo2_ge90','measured_lactate_lt2']:
                b=observed[observed.component_state.eq(3)]
                if subgroup=='all_spo2_ge90':b=b[b.spo2_min.ge(90)]
                elif subgroup=='measured_lactate_lt2':b=b[b.baseline_lactate.lt(2)]
                sync=b[b.synchronous_any.eq(1)];async_group=b[b.synchronous_any.eq(0)]
                es=int(sync[endpoint].sum());ea=int(async_group[endpoint].sum())
                nes=len(sync)-es;nea=len(async_group)-ea
                sufficient=es+ea>=100 and min(es,ea,nes,nea)>=20
                gates.append(dict(dataset=d,variant=variant,endpoint=endpoint,subgroup=subgroup,
                    jointly_observed_with_outcome=len(observed),primary_comparison_n=len(b),
                    events=es+ea,synchronous_n=len(sync),synchronous_events=es,
                    asynchronous_n=len(async_group),asynchronous_events=ea,
                    passes_count_gate=bool(sufficient),
                    decision='eligible for model support checks' if sufficient else 'counts only; frozen event gate failed'))
for name,obj in [('timing_contrast_support.json',gates),('component_state_counts.json',states),
                 ('endpoint_support_canonical.json',endpoint_support)]:
    (OUT/name).write_text(json.dumps(obj,indent=2,allow_nan=False)+'\n')
print(pd.DataFrame(gates).query("subgroup == 'all'").to_string(index=False))
con.close()
print('Support gate evaluated locally; no model fit or biological discovery asserted by this cell.')

dataset  variant           endpoint subgroup  jointly_observed_with_outcome  primary_comparison_n  events  synchronous_n  synchronous_events  asynchronous_n  asynchronous_events  passes_count_gate                              decision
   eicu   pooled hospital_mortality      all                           1013                   184      57             76                  24             108                   33              False counts only; frozen event gate failed
   eicu   pooled  troponin_rise_12h      all                            190                    43      29             19                  14              24                   15              False counts only; frozen event gate failed
   eicu   pooled  troponin_rise_24h      all                            191                    47      30             20                  14              27                   16              False counts only; frozen event gate failed
  mimic   pooled hospital_mortality      all                

In [9]:
# Conditional association screen; it runs only when the frozen gates pass.
import json, importlib.util, subprocess, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import duckdb
OUT=PROJECT/'research'/'spo2_circulatory_context'
gates=json.loads((OUT/'timing_contrast_support.json').read_text())
primary=[g for g in gates if g['variant']=='pooled' and g['subgroup']=='all'
         and g['endpoint'] in ['hospital_mortality','troponin_rise_12h']]
results=[]
if any(g['passes_count_gate'] for g in primary):
    if importlib.util.find_spec('statsmodels') is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q','statsmodels==0.14.4','scipy==1.15.2'])
    import statsmodels.api as sm
    from statsmodels.tools.sm_exceptions import PerfectSeparationWarning
    from statsmodels.stats.multitest import multipletests
    con=duckdb.connect(str(DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'/'spo2_context.duckdb'))
for gate in primary:
    result=dict(dataset=gate['dataset'],endpoint=gate['endpoint'],stage='conditional association screen',
                causal=False,treatment_adjusted=False,biological_discovery=False)
    if not gate['passes_count_gate']:
        result.update(status='not fitted',reason='frozen count gate failed');results.append(result);continue
    d=gate['dataset'];endpoint=gate['endpoint']
    f=con.execute(f'SELECT * FROM {d}_pooled_features WHERE joint_eligible AND joint_spo2_jump_any=1 AND map_fall_any=1').df()
    e=con.execute(f'SELECT * FROM {d}_context_endpoints').df()
    a=f.merge(e,on='stay_id',validate='one_to_one').dropna(subset=[endpoint]).copy()
    numeric=['age','hr_mean','spo2_mean','spo2_min','map_mean','map_min',
             'joint_spo2_jumps','map_falls','joint_transitions']
    a['hr_missing']=a.hr_mean.isna().astype(float)
    a['hr_mean']=a.hr_mean.fillna(a.hr_mean.median())
    a=a.dropna(subset=numeric+['spo2_readings','person_id'])
    x=a[numeric].astype(float).copy()
    x['log_spo2_readings']=np.log1p(a.spo2_readings)
    for col in x:
        sd=x[col].std(ddof=0)
        x[col]=(x[col]-x[col].mean())/sd if sd>0 else 0.0
    sex=a.is_male.map({0.0:'female',1.0:'male'}).fillna('unknown')
    x=pd.concat([x,pd.get_dummies(sex,prefix='sex',drop_first=True,dtype=float)],axis=1)
    x['hr_missing']=a.hr_missing
    x=x.loc[:,x.nunique()>1]
    x=sm.add_constant(x,has_constant='add');x['synchronous_any']=a.synchronous_any.astype(float)
    y=a[endpoint].astype(float)
    grouped=a.groupby('synchronous_any')[endpoint].agg(['count','sum'])
    event_gate=len(grouped)==2 and y.sum()>=100 and grouped['sum'].min()>=20 and (grouped['count']-grouped['sum']).min()>=20
    result.update(n=len(a),events=int(y.sum()),coefficients=x.shape[1],person_clusters=int(a.person_id.nunique()))
    reason=None
    if not event_gate:reason='count gate failed after required-covariate exclusions'
    elif min(y.sum(),len(y)-y.sum())<10*x.shape[1]:reason='fewer than 10 events or non-events per coefficient'
    elif a.person_id.nunique()<30:reason='fewer than 30 independent person clusters'
    elif np.linalg.matrix_rank(x.to_numpy())<x.shape[1]:reason='rank deficient design'
    if reason:
        result.update(status='not fitted',reason=reason);results.append(result);continue
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('error',PerfectSeparationWarning)
            base=sm.GLM(y,x.drop(columns='synchronous_any'),family=sm.families.Binomial()).fit(maxiter=100)
            model=sm.GLM(y,x,family=sm.families.Binomial()).fit(maxiter=100,
                cov_type='cluster',cov_kwds={'groups':a.person_id})
        if not base.converged or not model.converged:raise ValueError('Model did not converge')
        beta=float(model.params['synchronous_any']);se=float(model.bse['synchronous_any'])
        if not np.isfinite([beta,se]).all() or se<=0:raise ValueError('Invalid clustered uncertainty')
        if max(abs(beta-1.96*se),abs(beta+1.96*se))>700:raise ValueError('Unstable, unbounded effect interval')
        result.update(status='fitted',conditional_odds_ratio=float(np.exp(beta)),
            ci95=[float(np.exp(beta-1.96*se)),float(np.exp(beta+1.96*se))],
            p_value=float(model.pvalues['synchronous_any']),
            base_deviance=float(base.deviance),extended_deviance=float(model.deviance))
        if d=='eicu' and a.hospital_id.nunique()>=30:
            hospital=sm.GLM(y,x,family=sm.families.Binomial()).fit(maxiter=100,
                cov_type='cluster',cov_kwds={'groups':a.hospital_id})
            hs=float(hospital.bse['synchronous_any'])
            result['hospital_cluster_ci95']=[float(np.exp(beta-1.96*hs)),float(np.exp(beta+1.96*hs))]
    except (ValueError,np.linalg.LinAlgError,PerfectSeparationWarning) as exc:
        result.update(status='fit failed',reason=str(exc))
    results.append(result)
if any(g['passes_count_gate'] for g in primary):
    con.close()
    for d in ['eicu','mimic']:
        rows=[r for r in results if r['dataset']==d]
        adjusted=multipletests([r.get('p_value',1.0) if r['status']=='fitted' else 1.0 for r in rows],method='holm')[1]
        for r,p in zip(rows,adjusted):
            if r['status']=='fitted':r['holm_p_two_primary_endpoints']=float(p)
(OUT/'conditional_screen_results.json').write_text(json.dumps(results,indent=2,allow_nan=False)+'\n')
replicated=[]
for endpoint in ['hospital_mortality','troponin_rise_12h']:
    pair=[r for r in results if r['endpoint']==endpoint]
    if len(pair)==2 and all(r['status']=='fitted' and r['conditional_odds_ratio']>1
                          and r.get('holm_p_two_primary_endpoints',1)<.05 for r in pair):
        replicated.append(endpoint)
from datetime import datetime,timezone
import hashlib
finished=datetime.now(timezone.utc).isoformat()
manifest=dict(environment='google_colab' if IN_COLAB else 'local',finished_utc=finished,
    analysis_lock_sha256=ANALYSIS_LOCK_SHA256,synthetic_patients=0,
    raw_input_manifest_sha256=hashlib.sha256((OUT/'input_manifest.json').read_bytes()).hexdigest(),
    implementation_sha256=hashlib.sha256((PROJECT/'scripts/build_biological_notebook.py').read_bytes()).hexdigest(),
    production_spo2_helper_sha256=hashlib.sha256((PROJECT/'src/physiograph/analysis/spo2_protocol.py').read_bytes()).hexdigest(),
    software=dict(python=sys.version,numpy=np.__version__,pandas=pd.__version__,duckdb=duckdb.__version__),
    cross_database_positive_conditional_screens=replicated,
    paradigm_shifting_result=False,clinical_benefit_established=False)
(OUT/'analysis_manifest.json').write_text(json.dumps(manifest,indent=2)+'\n')
(OUT/'run_status.json').write_text(json.dumps(dict(stage='SpO2 circulatory-context candidate analyzed',
    status='complete',environment=manifest['environment'],updated_utc=finished,goal_complete=False,
    cross_database_positive_conditional_screens=replicated),indent=2)+'\n')
print(pd.DataFrame(results).to_string(index=False))
print('This is a conditional association screen. The biological discovery goal is not complete.')

dataset           endpoint                          stage  causal  treatment_adjusted  biological_discovery     status                   reason
   eicu hospital_mortality conditional association screen   False               False                 False not fitted frozen count gate failed
   eicu  troponin_rise_12h conditional association screen   False               False                 False not fitted frozen count gate failed
  mimic hospital_mortality conditional association screen   False               False                 False not fitted frozen count gate failed
  mimic  troponin_rise_12h conditional association screen   False               False                 False not fitted frozen count gate failed
This is a conditional association screen. The biological discovery goal is not complete.


# Biological direction of the SpO2–troponin association

Prospective specification for a new mechanistic falsification analysis,
2026-09-05 UTC. Patient data are restricted to MIMIC and eICU and the
original PhysioGraph HF cohorts. This is not a replacement threshold for
the failed MAP-synchrony candidate.

The current review's strongest two-source secondary signal is troponin
rise, but its advanced estimates disagree. The biological ordering of
myocardial injury and the first SpO2 episode remains unresolved. Delayed
troponin detection can follow injury that began before the respiratory
signal. Troponin kinetics do not establish infarction mechanism:
[primary clinical kinetics study](https://pmc.ncbi.nlm.nih.gov/articles/8360674/).

## First gate: can the biological ordering be observed?

Retain the production first absolute SpO2 change of at least four percentage
points, in 15-minute medians over [0,240) ICU minutes, with at least three
bins and two transitions separated by no more than 30 minutes. Use the
warning-filtered MIMIC replay and the current eICU periodic data. Do not
condition this question on having MAP measurements.

Collect troponin T and I measurements from the same hospital encounter from
24 hours before ICU admission through 28 hours afterward. Do not include
samples before the hospital encounter began. Preserve assay, unit, specimen
time and result/revision time; the latter are not interchangeable. Reuse
verified extracted post-admission samples and obtain only the missing
pre-admission-to-ICU segment from the original laboratory sources.

At the first SpO2 episode, examine the preceding 12 hours and following
12 hours. A serial pre-episode trajectory requires at least two positive
values of the same assay and unit separated by at least 60 minutes.
Choose the assay/unit with the earliest pre-episode measurement, breaking
ties by troponin T before I and then lexical unit. This selection uses
pre-episode data only. Define pre-episode rise as last/first >=1.5, preserving
the project's existing relative-rise threshold. Record the continuous
ratio as well. A post-episode trajectory requires a specimen at least
120 minutes after the episode and no later than 12 hours after it. The
post-episode ratio uses its maximum and the last pre-episode concentration.

First report only the numbers with an episode, pre-episode assays, serial
pre-episode measurements, and delayed post-episode measurements. Do not
inspect trajectory direction or join mortality in this first gate. At
least 100 exposed encounters with complete pre/post trajectories in EACH
database are required before advancing to direction estimates. If this
gate fails, retain the counts and stop this mechanistic candidate; do not
relax the separation, extend the window, or call one pre-episode sample a
stable trajectory.

If the gate passes, freeze the final directional contrast, uncertainty,
clinical controls, and mortality analysis before opening those outcomes.
Within-person temporal ordering alone cannot establish a treatment effect
or a paradigm-shifting biological discovery. An absent observed pre-episode
rise cannot rule out injury because biomarker appearance is delayed.

## Execution and reporting

Keep the runnable workflow in the existing single primary notebook.
Private patient-level tables remain under Data outside the repository.
Record source fingerprints, exact time/assay rules, exclusions and all failed
support gates. Label local runs accurately. No mortality improvement or
novel biological mechanism is presumed.


## Encounter-boundary amendment, before trajectory directions

The first feasibility pass used MIMIC `admittime` as encounter start and
found 17 complete trajectories. A source check then found that 987 of the
1,398 exposed encounters had an earlier `edregtime`, with a median 190
minutes from ED registration to inpatient admission. The [official MIMIC
documentation](https://mimic.mit.edu/docs/iv/modules/hosp/admissions.html)
distinguishes ED registration from inpatient admission.

For the revised feasibility pass, define MIMIC encounter start as the earlier
of `edregtime` and `admittime` from the same `hadm_id` row. Continue to require
an explicit matching laboratory `hadm_id`; do not assign unlinked samples by
proximity or by patient identity alone. Retain the original 24-hour pre-ICU
extraction bound, 12-hour pre/post-episode windows, 60-minute pre-sample
separation, 120-minute post-sample delay, assay-selection rule and 100-complete-
trajectory gate in both databases. eICU's hospital boundary is unchanged.

This amendment follows inspection of feasibility counts and source metadata,
but precedes inspection of trajectory directions, ratios or mortality.
Preserve the original specification, lock and results. This is a corrected
feasibility assessment, not a claim that the ED rule was prospectively frozen
before the first support pass or that a failed significance test was rescued.

In [10]:
DIRECTION_PROTOCOL_SHA256='cb137c6444593ddfa99eff90fd85c0216f92750c33f8f9c0d25290774f140490'
DIRECTION_BOUNDARY_SHA256='e9105d2d2973387c0181882f02e006d284d297a70efa73f4d34dc5b6b3e1fc2b'

In [11]:
# Biological-ordering feasibility; no trajectory ratios or mortality associations.
import sys,json,importlib.util
from pathlib import Path
from datetime import datetime,timezone
import duckdb
import pandas as pd
IN_COLAB=importlib.util.find_spec('google.colab') is not None if importlib.util.find_spec('google') else False
DRIVE=Path('/content/drive/MyDrive') if IN_COLAB else Path.home()/'Library/CloudStorage/GoogleDrive-2arnavmana@gmail.com/My Drive'
PROJECT=DRIVE/'Projects'/'PhysioGraph'
sys.path.insert(0,str(PROJECT/'src'))
from physiograph.analysis.spo2_protocol import _first_instability
from physiograph.etl.mimic_extractor import _mimic_lab_item_map
from physiograph.constants import EICU_LAB_NAME_MAP
PRIVATE=DRIVE/'Data'/'PhysioGraph_Biological_Discovery_20260905'
OUT=PROJECT/'research'/'spo2_troponin_direction';OUT.mkdir(exist_ok=True)
PRIOR=PROJECT/'research'/'spo2_circulatory_context'
def direction_json(name,obj):
    (OUT/name).write_text(json.dumps(obj,indent=2,allow_nan=False,default=str)+'\n')
def direction_status(stage,status='running'):
    direction_json('run_status.json',dict(stage=stage,status=status,
        environment='google_colab' if IN_COLAB else 'local',
        updated_utc=datetime.now(timezone.utc).isoformat(),goal_complete=False))
    print(stage,status,flush=True)
lock=OUT/'protocol_lock.json'
if lock.exists():assert json.loads(lock.read_text())['sha256']==DIRECTION_PROTOCOL_SHA256
else:direction_json('protocol_lock.json',dict(sha256=DIRECTION_PROTOCOL_SHA256,
    locked_before_support_extraction=True,utc=datetime.now(timezone.utc).isoformat()))
amendment=OUT/'boundary_amendment.json'
if amendment.exists():assert json.loads(amendment.read_text())['sha256']==DIRECTION_BOUNDARY_SHA256
else:direction_json('boundary_amendment.json',dict(sha256=DIRECTION_BOUNDARY_SHA256,
    initial_support_counts_already_seen=True,trajectory_directions_not_seen=True,
    utc=datetime.now(timezone.utc).isoformat()))
con=duckdb.connect(str(PRIVATE/'troponin_direction.duckdb'))
con.execute('SET threads=2');con.execute("SET memory_limit='2GB'")
def qs(value):return "'"+str(value).replace("'","''")+"'"
con.execute(f"ATTACH {qs(PRIVATE/'spo2_context.duckdb')} AS prior (READ_ONLY)")
assert con.execute("SELECT count(*) FROM information_schema.tables WHERE table_catalog='prior' AND table_name='mimic_context_quality_marker'").fetchone()[0]>0
manifest=json.loads((PRIOR/'input_manifest.json').read_text())
used={}
def verified_source(dataset,name):
    key=dataset+'/'+name;info=manifest[key];p=Path(info['path']);st=p.stat()
    assert st.st_size==info['bytes'] and st.st_mtime_ns==info['mtime_ns'], 'Source changed: '+key
    used[key]=info
    return p
direction_status('episode anchors and hospital boundaries')
coverage=[]
for d,source in [('eicu','eICU'),('mimic','MIMIC')]:
    fields='stay_id,followup_end_offset_minutes'+(',admit_time,hadm_id' if d=='mimic' else '')
    c=con.execute(f'SELECT {fields} FROM prior.{d}_cohort').df()
    raw=con.execute(f"""SELECT v.stay_id,'{d}' dataset,concept,
        event_minute offset_minutes,value value_numeric FROM prior.{d}_vitals v
        JOIN prior.{d}_cohort c USING(stay_id) WHERE concept='spo2'""").df()
    onset=_first_instability(raw).rename(columns={'instability_onset_minutes':'episode_minute'})
    a=c.merge(onset[['stay_id','episode_minute']],on='stay_id',validate='one_to_one')
    if d=='eicu':
        patient=pd.read_csv(verified_source(source,'patient'),usecols=['patientunitstayid','hospitaladmitoffset'])
        starts=patient.set_index('patientunitstayid').hospitaladmitoffset
        a['hospital_start_minute']=pd.to_numeric(a.stay_id.map(starts),errors='coerce')
    else:
        admission=pd.read_csv(verified_source(source,'admissions'),usecols=['hadm_id','admittime','edregtime']).set_index('hadm_id')
        clocks=admission[['admittime','edregtime']].apply(pd.to_datetime,errors='coerce')
        starts=clocks.min(axis=1)
        a['hospital_start_minute']=(a.hadm_id.map(starts)-a.admit_time).dt.total_seconds()/60.0
    missing=int(a.hospital_start_minute.isna().sum())
    a=a[a.hospital_start_minute.notna()].copy()
    con.register('episode_anchors',a)
    con.execute(f'CREATE OR REPLACE TABLE {d}_anchors AS SELECT * FROM episode_anchors')
    coverage.append(dict(dataset=d,cohort=len(c),episodes=len(onset),
        missing_hospital_start=missing,anchors=len(a)))
direction_json('episode_support.json',coverage)
direction_status('eICU pre-ICU troponin extraction')
mapping=pd.DataFrame([dict(raw_name=k,concept=v) for k,v in EICU_LAB_NAME_MAP.items() if v in ['troponin_t','troponin_i']])
con.register('assay_map',mapping)
lab=verified_source('eICU','lab')
con.execute(f"""CREATE OR REPLACE TABLE eicu_preicu_troponin AS
 SELECT a.stay_id,try_cast(l.labresultoffset AS DOUBLE) event_minute,
 try_cast(l.labresultrevisedoffset AS DOUBLE) available_minute,m.concept,
 lower(trim(coalesce(l.labmeasurenamesystem,l.labmeasurenameinterface,''))) unit,
 try_cast(l.labresult AS DOUBLE) AS value
 FROM read_csv({qs(lab)},header=true,all_varchar=true,sample_size=10000,strict_mode=true) l
 JOIN eicu_anchors a ON try_cast(l.patientunitstayid AS BIGINT)=a.stay_id
 JOIN assay_map m ON lower(trim(l.labname))=m.raw_name
 WHERE try_cast(l.labresultoffset AS DOUBLE)>=greatest(-1440,a.hospital_start_minute)
 AND try_cast(l.labresultoffset AS DOUBLE)<0
 AND try_cast(l.labresult AS DOUBLE) IS NOT NULL""")
direction_status('MIMIC pre-ICU troponin extraction')
root=DRIVE/'Data'/'MIMIC'/'Full'
mapping=pd.DataFrame([dict(itemid=k,concept=v) for k,v in _mimic_lab_item_map(root).items() if v in ['troponin_t','troponin_i']])
con.register('assay_map',mapping)
lab=verified_source('MIMIC','labevents');verified_source('MIMIC','d_labitems')
con.execute(f"""CREATE OR REPLACE TABLE mimic_preicu_troponin AS
 SELECT a.stay_id,date_diff('second',a.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0 event_minute,
 date_diff('second',a.admit_time,try_cast(l.storetime AS TIMESTAMP))/60.0 available_minute,
 m.concept,lower(trim(coalesce(l.valueuom,''))) unit,try_cast(l.valuenum AS DOUBLE) AS value
 FROM read_csv({qs(lab)},header=true,all_varchar=true,sample_size=10000,strict_mode=true) l
 JOIN mimic_anchors a ON try_cast(l.hadm_id AS BIGINT)=a.hadm_id
 JOIN assay_map m ON try_cast(l.itemid AS BIGINT)=m.itemid
 WHERE date_diff('second',a.admit_time,try_cast(l.charttime AS TIMESTAMP))/60.0>=greatest(-1440,a.hospital_start_minute)
 AND try_cast(l.charttime AS TIMESTAMP)<a.admit_time AND try_cast(l.valuenum AS DOUBLE) IS NOT NULL""")
for key,info in used.items():
    st=Path(info['path']).stat()
    assert st.st_size==info['bytes'] and st.st_mtime_ns==info['mtime_ns'], 'Source changed during scan: '+key
direction_json('source_manifest.json',dict(verified_fingerprints_reused_from=str(PRIOR/'input_manifest.json'),
    same_size_and_mtime_before_and_after=True,sources=used))
direction_status('serial troponin trajectory support')
support=[]
for d in ['eicu','mimic']:
    con.execute(f"""CREATE OR REPLACE TABLE {d}_troponin AS
      WITH combined AS (
        SELECT stay_id,event_minute,available_minute,concept,unit,value FROM {d}_preicu_troponin
        UNION ALL
        SELECT l.stay_id,event_minute,available_minute,concept,unit,value FROM prior.{d}_context_labs l
        JOIN {d}_anchors a USING(stay_id) WHERE concept IN ('troponin_t','troponin_i'))
      SELECT stay_id,event_minute,concept,unit,median(value) AS value,max(available_minute) available_minute
      FROM combined WHERE value>0 AND value<=1000000 GROUP BY stay_id,event_minute,concept,unit""")
    con.execute(f"""CREATE OR REPLACE TABLE {d}_trajectory_support AS
      WITH pre_assays AS (
        SELECT t.stay_id,t.concept,t.unit,count(*) pre_n,min(event_minute) pre_first_time,
        max(event_minute) pre_last_time FROM {d}_troponin t JOIN {d}_anchors a USING(stay_id)
        WHERE event_minute>=a.episode_minute-720 AND event_minute<a.episode_minute
          AND event_minute>=a.hospital_start_minute
        GROUP BY t.stay_id,t.concept,t.unit), chosen AS (
        SELECT * FROM pre_assays QUALIFY row_number() OVER(PARTITION BY stay_id
          ORDER BY pre_first_time,CASE WHEN concept='troponin_t' THEN 0 ELSE 1 END,unit)=1), post_assays AS (
        SELECT c.stay_id,count(t.event_minute) post_n FROM chosen c JOIN {d}_anchors a USING(stay_id)
        LEFT JOIN {d}_troponin t ON t.stay_id=c.stay_id AND t.concept=c.concept AND t.unit=c.unit
          AND t.event_minute>=a.episode_minute+120 AND t.event_minute<=a.episode_minute+720
        GROUP BY c.stay_id)
      SELECT a.stay_id,a.episode_minute,c.concept,c.unit,c.pre_n,c.pre_first_time,c.pre_last_time,
        coalesce(p.post_n,0) post_n,
        coalesce(c.pre_n>=2 AND c.pre_last_time-c.pre_first_time>=60,FALSE) serial_pre,
        coalesce(c.pre_n>=2 AND c.pre_last_time-c.pre_first_time>=60 AND p.post_n>=1,FALSE) complete_trajectory
      FROM {d}_anchors a LEFT JOIN chosen c USING(stay_id) LEFT JOIN post_assays p USING(stay_id)""")
    s=con.execute(f'SELECT * FROM {d}_trajectory_support').df()
    support.append(dict(dataset=d,episodes_with_hospital_boundary=len(s),
        any_pre_episode_assay=int(s.pre_n.notna().sum()),serial_pre_episode=int(s.serial_pre.sum()),
        complete_pre_post_trajectories=int(s.complete_trajectory.sum()),
        passes_count_gate=bool(s.complete_trajectory.sum()>=100)))
direction_json('trajectory_support.json',support)
direction_json('analysis_manifest.json',dict(environment='google_colab' if IN_COLAB else 'local',
    protocol_sha256=DIRECTION_PROTOCOL_SHA256,boundary_amendment_sha256=DIRECTION_BOUNDARY_SHA256,
    completed_utc=datetime.now(timezone.utc).isoformat(),
    all_sources_pass=all(s['passes_count_gate'] for s in support),
    trajectory_directions_inspected=False,mortality_joined=False,synthetic_patients=0,
    biological_discovery=False))
con.close()
print(pd.DataFrame(support).to_string(index=False))
direction_status('biological-ordering feasibility','complete')

episode anchors and hospital boundaries running
eICU pre-ICU troponin extraction running
MIMIC pre-ICU troponin extraction running
serial troponin trajectory support running
dataset  episodes_with_hospital_boundary  any_pre_episode_assay  serial_pre_episode  complete_pre_post_trajectories  passes_count_gate
   eicu                             4309                   1326                 224                             133               True
  mimic                             1398                    288                  40                              23              False
biological-ordering feasibility complete
